# 🦋 Grounding DINO + Qwen3-VL

Последовательность: DINO → Qwen → DINO для цветов → очистка → метрики. Для каждой модели обрабатываются 40 размеченных и 400 основных фотографий. Эталон читается после построения предсказаний. Требуется архив `butterfly_dataset.zip` с `photos.json`, `golden.jsonl`, `images/labeled/` и `images/unlabeled/`.

В Colab включите T4 GPU и запускайте ячейки по порядку. Две модели загружаются по очереди. Скачайте результаты до завершения сессии: `/content` временный. Это предсказания, требующие проверки человеком.

## 1. Установка и настройки

In [16]:
%pip -q install "transformers==5.5.0" "huggingface-hub>=1.32,<2.0" accelerate "openpyxl>=3.1" pillow tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


In [17]:
import gc, json, re, time, zipfile
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Qwen3VLForConditionalGeneration
try:
    from google.colab import files
except ImportError:
    files = None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)
if device == 'cpu': print('Для скорости рекомендуется T4 GPU.')

Устройство: cuda


In [18]:
# Выберите путь к уже загруженному архиву или загрузку с компьютера.
ZIP_SOURCE = 'path'  # 'path' или 'upload'

DATASET_ZIP = Path('/kaggle/working/butterfly_dataset_new.zip')

print('Архив существует:', DATASET_ZIP.is_file())
print('Путь:', DATASET_ZIP)
if ZIP_SOURCE == 'upload':
    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zips) != 1: raise ValueError('Выберите ровно один ZIP.')
    DATASET_ZIP = Path(zips[0])
elif ZIP_SOURCE != 'path':
    raise ValueError("ZIP_SOURCE должен быть 'path' или 'upload'.")

OUT = Path('/kaggle/working/butterfly_models_results')
OUT.mkdir(parents=True, exist_ok=True)
DINO_MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
DINO_PROMPT = 'an adult butterfly.'
FLOWER_PROMPT = 'flower.'
DINO_BOX_THRESHOLD = 0.40
FLOWER_BOX_THRESHOLD = 0.25
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.50
MATCH_IOU_THRESHOLD = 0.90
SCORE_MARGIN = 0.10
QWEN_PROMPT = 'Locate every adult butterfly and moths in the image. Exclude caterpillars, flowers and leaves. Return only a JSON array, one object per butterfly, in the form [{"bbox_2d": [x_min, y_min, x_max, y_max], "label": "butterfly"}]. Coordinates must be integers from 0 to 1000 relative to the full image. If there are no butterflies, return [].'
MAX_NEW_TOKENS = 512

Архив существует: True
Путь: /kaggle/working/butterfly_dataset_new.zip


In [19]:
from pathlib import Path
import json
import zipfile

# Папка с НОВЫМ датасетом из панели Input
INPUT_DIR = Path(
    '/kaggle/input/datasets/burkovanastia/'
    'new-butterflies/butterfly_dataset_new'
)

# Сюда создадим ZIP, который прочитает ноутбук
DATASET_ZIP = Path('/kaggle/working/butterfly_dataset_new.zip')

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f'Папка не найдена: {INPUT_DIR}\n'
        'Скопируй её путь из панели Input справа и вставь в INPUT_DIR.'
    )

# Ищем файлы именно внутри нового датасета.
manifests = [
    path
    for path in INPUT_DIR.rglob('photos.json')
    if (path.parent / 'golden.jsonl').is_file()
    and (path.parent / 'images').is_dir()
]

if len(manifests) != 1:
    raise ValueError(
        f'Ожидалась одна папка с photos.json, golden.jsonl и images, '
        f'найдено: {len(manifests)}.\n'
        f'Результаты: {manifests}'
    )

dataset_dir = manifests[0].parent
print('Найден новый датасет:', dataset_dir)

# Проверяем количество фотографий в описании набора.
with open(dataset_dir / 'photos.json', encoding='utf-8-sig') as file:
    manifest = json.load(file)

print('Основных фотографий:', len(manifest.get('unlabeled', [])))
print('Эталонных фотографий:', len(manifest.get('labeled', [])))

# Собираем ZIP без повторного сжатия фотографий — так быстрее.
with zipfile.ZipFile(
    DATASET_ZIP,
    mode='w',
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as archive:

    archive.write(dataset_dir / 'photos.json', 'photos.json')
    archive.write(dataset_dir / 'golden.jsonl', 'golden.jsonl')

    image_files = [
        path
        for path in (dataset_dir / 'images').rglob('*')
        if path.is_file()
    ]

    for i, image_path in enumerate(image_files, start=1):
        relative_path = image_path.relative_to(dataset_dir)
        archive.write(image_path, relative_path.as_posix())

        if i % 100 == 0 or i == len(image_files):
            print(f'Добавлено файлов: {i}/{len(image_files)}')

print('\nАрхив создан:', DATASET_ZIP)
print('Теперь можно запускать ячейку «Проверка архива».')

Найден новый датасет: /kaggle/input/datasets/burkovanastia/new-butterflies/butterfly_dataset_new
Основных фотографий: 400
Эталонных фотографий: 40
Добавлено файлов: 100/440
Добавлено файлов: 200/440
Добавлено файлов: 300/440
Добавлено файлов: 400/440
Добавлено файлов: 440/440

Архив создан: /kaggle/working/butterfly_dataset_new.zip
Теперь можно запускать ячейку «Проверка архива».


## 2. Проверка архива

In [20]:
def read_kaggle_manifest(dataset_zip):
    if not dataset_zip.is_file():
        raise FileNotFoundError(f'Архив не найден: {dataset_zip}. Укажи правильный DATASET_ZIP.')
    with zipfile.ZipFile(dataset_zip) as z:
        names = z.namelist()
        manifests = [n for n in names if n.rsplit('/', 1)[-1] == 'photos.json' and not n.startswith('__MACOSX/')]
        if len(manifests) != 1:
            raise ValueError(f'Ожидался один photos.json; найдено: {len(manifests)}')
        root = manifests[0][:-len('photos.json')]
        manifest = json.loads(z.read(manifests[0]).decode('utf-8-sig'))
        photos = []
        for subset, expected in [('labeled', 40), ('unlabeled', 400)]:
            rows = manifest.get(subset, [])
            if len(rows) != expected:
                raise ValueError(f'{subset}: ожидается {expected} фотографий, найдено {len(rows)}')
            for row in rows:
                relative = str(row['file'])
                if '..' in Path(relative).parts or relative.startswith('/'):
                    raise ValueError(f'Недопустимый путь в manifest: {relative}')
                member = root + relative
                if member not in names:
                    raise FileNotFoundError(f'Не найдено в архиве: {member}')
                photos.append({'photo_id': str(row['photo_id']), 'subset': subset, 'member': member})
        if len({p['photo_id'] for p in photos}) != len(photos):
            raise ValueError('В данных обнаружены повторяющиеся photo_id.')
        if root + 'golden.jsonl' not in names:
            raise FileNotFoundError('Не найден golden.jsonl рядом с photos.json.')
    return photos, root

photos, dataset_root = read_kaggle_manifest(DATASET_ZIP)
print('Фотографий:', len(photos), '; для эталона:', sum(p['subset'] == 'labeled' for p in photos))

Фотографий: 440 ; для эталона: 40


## 3. Общие функции

In [21]:
def open_image(archive, member):
    with Image.open(BytesIO(archive.read(member))) as img:
        return ImageOps.exif_transpose(img).convert('RGB')

def nms_indices(boxes, scores, iou_threshold):
    # NMS: из пересекающихся рамок сохраняем рамку с наибольшим score.
    if len(boxes) == 0:
        return []
    boxes = np.asarray(boxes, dtype=float)
    order = np.argsort(-np.asarray(scores, dtype=float))
    keep = []
    while len(order):
        i = int(order[0])
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        lt = np.maximum(boxes[i, :2], boxes[rest, :2])
        rb = np.minimum(boxes[i, 2:], boxes[rest, 2:])
        wh = np.maximum(rb - lt, 0)
        intersection = wh[:, 0] * wh[:, 1]
        area_i = np.prod(np.maximum(boxes[i, 2:] - boxes[i, :2], 0))
        area_rest = np.prod(np.maximum(boxes[rest, 2:] - boxes[rest, :2], 0), axis=1)
        iou = intersection / np.maximum(area_i + area_rest - intersection, 1e-9)
        order = rest[iou <= iou_threshold]
    return keep


def run_inference(name, predict, include_raw=False):
    records = []
    subset_counter = {'labeled': 0, 'unlabeled': 0}
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        for index, p in enumerate(tqdm(photos, desc=name, unit='фото')):
            subset_counter[p['subset']] += 1
            row = {'photo_id': p['photo_id'], 'subset': p['subset'], 'member': p['member'],
                   'image_number': subset_counter[p['subset']], 'boxes': None, 'width': None, 'height': None,
                   'seconds': 0.0, 'error': None, 'raw_response': ''}
            try:
                image = open_image(archive, p['member'])
                row['width'], row['height'] = image.size
                if torch.cuda.is_available(): torch.cuda.synchronize()
                start = time.perf_counter()
                prediction = predict(image)
                if torch.cuda.is_available(): torch.cuda.synchronize()
                row['seconds'] = time.perf_counter() - start
                if include_raw: row['boxes'], row['raw_response'] = prediction
                else: row['boxes'] = prediction
            except Exception as exc:
                row['error'] = f'{type(exc).__name__}: {exc}'[:500]
                row['raw_response'] = getattr(exc, 'raw_text', '')
            records.append(row)
    error_count = sum(r['error'] is not None for r in records)
    print(f'{name}: обработано {len(records)}; ошибок: {error_count}')
    return records

def save_records(name, records):
    # JSONL хранит пиксельные координаты и исходные ответы для диагностики.
    for subset, count in [('labeled', 40), ('unlabeled', 400)]:
        subset_rows = [r for r in records if r['subset'] == subset]
        (OUT/f'{name}_{count}.jsonl').write_text(
            ''.join(json.dumps(r, ensure_ascii=False, allow_nan=False)+'\n' for r in subset_rows), encoding='utf-8')
    rows = []
    for r in records:
        row = {k: v for k,v in r.items() if k != 'boxes'}
        row['status'] = 'ok' if r['error'] is None else 'error'
        row['boxes_json'] = json.dumps(r['boxes'], ensure_ascii=False) if r['boxes'] is not None else ''
        row['num_boxes'] = len(r['boxes']) if r['boxes'] is not None else None
        row['butterfly_detected'] = int(bool(r['boxes'])) if r['boxes'] is not None else None
        rows.append(row)
    table = pd.DataFrame(rows)
    with pd.ExcelWriter(OUT/f'{name}_400.xlsx', engine='openpyxl') as writer:
        table[table['subset']=='unlabeled'].to_excel(writer, sheet_name='images', index=False)
    with pd.ExcelWriter(OUT/f'{name}_40.xlsx', engine='openpyxl') as writer:
        table[table['subset']=='labeled'].to_excel(writer, sheet_name='images', index=False)
    print(f'Сохранено: {name}_400.xlsx и {name}_40.xlsx')

def release_model(model, processor):
    del model, processor
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 4. Grounding DINO: бабочки

In [22]:
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_ID).to(device).eval()

def predict_dino(image, prompt, threshold):
    inputs = dino_processor(images=image, text=prompt, return_tensors='pt').to(device)
    with torch.inference_mode(): outputs = dino_model(**inputs)
    result = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=threshold, text_threshold=TEXT_THRESHOLD,
        target_sizes=[(image.height, image.width)])[0]
    boxes = result['boxes'].detach().cpu().numpy()
    scores = result['scores'].detach().cpu().numpy()
    predictions = []
    for i in nms_indices(boxes, scores, NMS_IOU_THRESHOLD):
        x1,y1,x2,y2 = boxes[i].tolist()
        x1,x2 = max(0,min(x1,image.width)),max(0,min(x2,image.width))
        y1,y2 = max(0,min(y1,image.height)),max(0,min(y2,image.height))
        if x2 > x1 and y2 > y1:
            predictions.append({'x_min':round(x1,2),'y_min':round(y1,2),'x_max':round(x2,2),
                                'y_max':round(y2,2),'confidence':round(float(scores[i]),4)})
    return predictions

dino_records = run_inference('DINO: бабочки', lambda image: predict_dino(image, DINO_PROMPT, DINO_BOX_THRESHOLD))
save_records('dino', dino_records)
del dino_model, dino_processor
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

DINO: бабочки:   0%|          | 0/440 [00:00<?, ?фото/s]

DINO: бабочки: обработано 440; ошибок: 0
Сохранено: dino_400.xlsx и dino_40.xlsx


## 5. Qwen3-VL: бабочки

In [23]:
processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None).eval()
TEXT_PROMPT = QWEN_PROMPT
class ModelOutputError(ValueError):
    def __init__(self, message, raw_text):
        super().__init__(message)
        self.raw_text = raw_text

def generate_prediction(image):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': TEXT_PROMPT},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    tokens = output[0][inputs['input_ids'].shape[-1]:]
    raw = processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    # Сохраняем raw, даже если ответ обрезан или невалиден.
    if len(tokens) >= MAX_NEW_TOKENS and tokens[-1].item() != processor.tokenizer.eos_token_id:
        raise ModelOutputError('Ответ достиг MAX_NEW_TOKENS и, возможно, обрезан.', raw)
    clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.IGNORECASE).strip()
    try:
        items = json.loads(clean)
    except (json.JSONDecodeError, TypeError) as exc:
        raise ModelOutputError(f'Ответ не является корректным JSON: {exc}', raw) from exc
    if not isinstance(items, list):
        raise ModelOutputError('Ожидался JSON-массив, например [] или [{"bbox_2d": [...]}].', raw)
    boxes = []
    for item in items:
        if not isinstance(item, dict) or not isinstance(item.get('bbox_2d'), list) or len(item['bbox_2d']) != 4:
            raise ModelOutputError('Некорректная запись bbox_2d.', raw)
        try:
            coords = [float(v) for v in item['bbox_2d']]
        except (TypeError, ValueError) as exc:
            raise ModelOutputError('Координаты должны быть числами.', raw) from exc
        if not all(np.isfinite(coords)) or not all(0 <= v <= 1000 for v in coords):
            raise ModelOutputError('Координаты должны быть в диапазоне 0–1000.', raw)
        x1, y1, x2, y2 = coords
        if x2 <= x1 or y2 <= y1:
            raise ModelOutputError('Некорректный порядок координат рамки.', raw)
        label = str(item.get('label', 'butterfly')).strip().lower()
        if label not in ('butterfly', 'adult butterfly'):
            raise ModelOutputError(f'Неожиданная метка {label!r}; нужна ручная проверка.', raw)
        boxes.append({'x_min': round(x1*image.width/1000, 2), 'y_min': round(y1*image.height/1000, 2),
                      'x_max': round(x2*image.width/1000, 2), 'y_max': round(y2*image.height/1000, 2),
                      'label': 'butterfly'})
    return boxes, raw


qwen_records = run_inference('Qwen: бабочки', generate_prediction, include_raw=True)
save_records('qwen', qwen_records)
del model, processor
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen: бабочки:   0%|          | 0/440 [00:00<?, ?фото/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen: бабочки: обработано 440; ошибок: 19
Сохранено: qwen_400.xlsx и qwen_40.xlsx


## 6. Эксперимент: DINO и очистка от цветов

DINO отдельно ищет цветы на всех 440 фото. Рамка бабочки удаляется только при IoU ≥ порога и большей оценке цветка с указанным отрывом. Это эвристика: оценки разных запросов могут быть несопоставимы, а бабочка может сидеть на цветке.

In [24]:
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_ID).to(device).eval()
flower_records = run_inference('DINO: цветы', lambda image: predict_dino(image, FLOWER_PROMPT, FLOWER_BOX_THRESHOLD))
save_records('flowers', flower_records)
del dino_model, dino_processor
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

DINO: цветы:   0%|          | 0/440 [00:00<?, ?фото/s]

DINO: цветы: обработано 440; ошибок: 0
Сохранено: flowers_400.xlsx и flowers_40.xlsx


In [25]:
def box_iou(a,b):
    w=max(0,min(a['x_max'],b['x_max'])-max(a['x_min'],b['x_min']))
    h=max(0,min(a['y_max'],b['y_max'])-max(a['y_min'],b['y_min']))
    inter=w*h
    aa=(a['x_max']-a['x_min'])*(a['y_max']-a['y_min'])
    bb=(b['x_max']-b['x_min'])*(b['y_max']-b['y_min'])
    return inter/(aa+bb-inter) if aa+bb-inter>0 else 0

if len(dino_records)!=len(flower_records): raise ValueError('Число записей не совпадает.')
clean_records, removed = [], []
for d, f in zip(dino_records, flower_records):
    if (d['photo_id'], d['subset']) != (f['photo_id'], f['subset']):
        raise ValueError('Порядок записей DINO и цветов не совпадает.')
    row = dict(d)
    row['seconds'] = d['seconds'] + f['seconds']
    if d['error'] or f['error']:
        row['boxes'] = None
        row['error'] = 'Проверка не выполнена: ' + str(d['error'] or f['error'])
    else:
        keep=[]
        for butterfly in d['boxes']:
            matches=[(box_iou(butterfly, flower), flower) for flower in f['boxes']
                     if box_iou(butterfly, flower)>=MATCH_IOU_THRESHOLD
                     and flower['confidence']>butterfly['confidence']
                     and flower['confidence']-butterfly['confidence']>=SCORE_MARGIN]
            if matches:
                overlap, flower=max(matches, key=lambda x: x[1]['confidence']-butterfly['confidence'])
                removed.append({'photo_id':d['photo_id'], 'subset':d['subset'], 'iou':overlap,
                                'butterfly':butterfly, 'flower':flower})
            else: keep.append(butterfly)
        row['boxes']=keep
        row['removed_count']=len(d['boxes'])-len(keep)
    clean_records.append(row)
save_records('dino_clean', clean_records)
(OUT/'removed_boxes.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in removed),encoding='utf-8')
print('Удалено рамок:', len(removed))

Сохранено: dino_clean_400.xlsx и dino_clean_40.xlsx
Удалено рамок: 14


## 7. Метрики на 40 фотографиях

In [26]:
# Метрики: та же логика максимального паросочетания, что в Kaggle.
def quality_xyxy(box):
    x, y = float(box['left']), float(box['top'])
    return [x, y, x + float(box['width']), y + float(box['height'])]

def quality_iou(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    low = np.maximum(a[:, None, :2], b[None, :, :2])
    high = np.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection = np.prod(np.maximum(high-low, 0), axis=2)
    area_a = np.prod(np.maximum(a[:, 2:]-a[:, :2], 0), axis=1)
    area_b = np.prod(np.maximum(b[:, 2:]-b[:, :2], 0), axis=1)
    return intersection / np.maximum(area_a[:, None]+area_b[None, :]-intersection, 1e-12)

def quality_matches(overlap, threshold):
    edges = overlap >= threshold
    owner = {}
    def assign(i, seen):
        for j in np.flatnonzero(edges[i]):
            j = int(j)
            if j in seen: continue
            seen.add(j)
            if j not in owner or assign(owner[j], seen):
                owner[j] = i
                return True
        return False
    return sum(assign(i, set()) for i in range(len(edges)))

def dino_metrics(golden_rows, predictions):
    gold = {str(row['photo_id']): row['boxes'] for row in golden_rows}
    ids = [str(row['photo_id']) for row in predictions]
    if len(gold) != len(golden_rows) or len(set(ids)) != len(ids) or set(gold) != set(ids):
        raise ValueError('ID предсказаний и эталона не совпадают или повторяются.')
    if any(row['error'] is not None or row['boxes'] is None for row in predictions):
        raise ValueError('В разметке есть ошибки; оценивать их как отсутствие бабочек нельзя.')
    expected = sum(len(v) for v in gold.values())
    predicted = sum(len(row['boxes']) for row in predictions)
    matches = {0.5: 0, 0.75: 0}
    for row in predictions:
        a = [quality_xyxy(b) for b in gold[str(row['photo_id'])]]
        b = [quality_xyxy(box) for box in row['boxes']]
        overlap = quality_iou(a, b)
        for threshold in matches: matches[threshold] += quality_matches(overlap, threshold)
    tp = matches[0.5]
    return {'F1@0.5': 2*tp/(expected+predicted) if expected+predicted else 0,
            'precision': tp/predicted if predicted else 0,
            'recall': tp/expected if expected else 0,
            'FP+FN': predicted+expected-2*tp,
            'time, s': sum(r['seconds'] for r in predictions),
            'F1@0.75': 2*matches[0.75]/(expected+predicted) if expected+predicted else 0,
            'FP on negatives': sum(len(r['boxes']) for r in predictions if not gold[str(r['photo_id'])])}


def to_normalized_for_metrics(records):
    output=[]
    for r in records:
        if r['subset']!='labeled': continue
        row={'photo_id':r['photo_id'],'seconds':r['seconds'],'error':r['error'],'boxes':None}
        if r['error'] is None:
            row['boxes']=[{'left':b['x_min']/r['width'], 'top':b['y_min']/r['height'],
                           'width':(b['x_max']-b['x_min'])/r['width'],
                           'height':(b['y_max']-b['y_min'])/r['height']} for b in r['boxes']]
        output.append(row)
    return output

with zipfile.ZipFile(DATASET_ZIP) as archive:
    golden = [json.loads(line) for line in archive.read(dataset_root+'golden.jsonl').decode('utf-8-sig').splitlines() if line.strip()]
if len(golden)!=40: raise ValueError('Ожидается 40 эталонных записей.')

metric_rows=[]
for name, records in [('DINO',dino_records),('Qwen',qwen_records),('DINO + очистка цветов',clean_records)]:
    labeled=to_normalized_for_metrics(records)
    errors=[r for r in labeled if r['error'] is not None]
    if errors:
        print(f'{name}: метрики не посчитаны — ошибок на 40 фото: {len(errors)}. Примеры:', errors[:2])
        metric_rows.append({'model':name,'status':f'ошибок: {len(errors)}'})
        continue
    values=dino_metrics(golden,labeled)
    values.update({'model':name,'status':'ok'})
    metric_rows.append(values)
metrics_df=pd.DataFrame(metric_rows).set_index('model')
display(metrics_df.round(3))
metrics_df.to_csv(OUT/'metrics.csv',encoding='utf-8-sig')
print('Время — сумма времени обработки 40 фото без загрузки изображений и весов; для очистки включает поиск цветов.')

,F1@0.5,precision,recall,FP+FN,"time, s",F1@0.75,FP on negatives,status
model,,,,,,,,
DINO,0.861,0.816,0.912,10,22.834,0.694,7,ok
Qwen,0.912,0.912,0.912,6,239.879,0.706,1,ok
DINO + очистка цветов,0.899,0.886,0.912,7,41.782,0.725,4,ok


Время — сумма времени обработки 40 фото без загрузки изображений и весов; для очистки включает поиск цветов.


## 8. Просмотр и скачивание

Номера в таблицах — порядок внутри каждого набора; в примере выбираем фотографию по `photo_id`. Результаты можно скачать отдельными файлами или архивом.

In [27]:
def show_results(photo_id, variant='dino'):
    variants={'dino':dino_records,'qwen':qwen_records,'dino_clean':clean_records,'flowers':flower_records}
    if variant not in variants: raise ValueError(f'variant: {list(variants)}')
    found=[r for r in variants[variant] if str(r['photo_id'])==str(photo_id)]
    if not found: print('Фото не найдено:',photo_id); return
    r=found[0]
    if r['error']: print('Ошибка:',r['error']); return
    with zipfile.ZipFile(DATASET_ZIP) as archive: image=open_image(archive,r['member'])
    fig,ax=plt.subplots(figsize=(10,7));ax.imshow(image)
    for box in r['boxes']:
        x1,y1,x2,y2=[box[k] for k in ['x_min','y_min','x_max','y_max']]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='red',linewidth=2))
        label=variant + (f" {box['confidence']:.2f}" if 'confidence' in box else '')
        ax.text(x1,y1,label,color='white',bbox=dict(facecolor='red',alpha=.8))
    ax.set_title(f"{variant} | {photo_id} | рамок: {len(r['boxes'])}");ax.axis('off');plt.show()

# Пример: замените ID на нужный из photos.json.
# show_results('758745658', variant='dino_clean')

In [28]:
settings={'dino_model':DINO_MODEL_ID,'qwen_model':QWEN_MODEL_ID,'dino_prompt':DINO_PROMPT,
          'qwen_prompt':QWEN_PROMPT,'flower_prompt':FLOWER_PROMPT,'dino_threshold':DINO_BOX_THRESHOLD,
          'flower_threshold':FLOWER_BOX_THRESHOLD,'text_threshold':TEXT_THRESHOLD,
          'nms_iou':NMS_IOU_THRESHOLD,'match_iou':MATCH_IOU_THRESHOLD,'score_margin':SCORE_MARGIN,
          'qwen_max_new_tokens':MAX_NEW_TOKENS,'device':device,'do_sample':False}
(OUT/'settings.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')
archive_path = Path('/kaggle/working/butterfly_models_results.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): z.write(path,path.name)
print('Результаты:',archive_path)

Результаты: /kaggle/working/butterfly_models_results.zip
